# Module 00: Setup and Profiling

Run every cell in order. At the end you should see a green **All checks passed** message.

## Cell 1: Install Dependencies

In [ ]:
# These are the only packages we need beyond what Kaggle pre-installs.
# This cell takes ~2 minutes on first run.
!pip install -q accelerate==0.34.0 deepspeed==0.14.0 transformers==4.44.0 datasets==2.21.0

## Cell 2: Verify Your Environment

In [ ]:
import torch
import subprocess
import sys

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
print()

total_vram = 0
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / 1e9
    total_vram += vram_gb
    print(f"  GPU {i}: {props.name}")
    print(f"    VRAM:          {vram_gb:.1f} GB")
    print(f"    CUDA cores:    {props.multi_processor_count * 128}")
    print(f"    SM count:      {props.multi_processor_count}")
    print()

print(f"Total VRAM: {total_vram:.1f} GB")

# Check
assert torch.cuda.is_available(), "ERROR: No CUDA. Did you enable GPU in Kaggle settings?"
assert torch.cuda.device_count() == 2, f"ERROR: Expected 2 GPUs, got {torch.cuda.device_count()}. Select GPU T4 x2 in Kaggle settings."
print("\n✓ All checks passed. You are ready to start.")

## Cell 3: Understand Ranks and World Size

Before writing distributed code, internalize these three concepts.

In [ ]:
# This is NOT distributed code yet. It just demonstrates the concepts.

# In distributed training, each GPU gets a unique 'rank'
# rank 0 is always the 'main' process (prints logs, saves checkpoints)

# Simulate what each process would print:
world_size = torch.cuda.device_count()  # 2

for simulated_rank in range(world_size):
    print(f"[Process rank={simulated_rank}] "
          f"I control GPU {simulated_rank}. "
          f"There are {world_size} processes total.")

print()
print("In real training, these print statements happen in parallel across GPUs.")
print("That is why you often see duplicate log lines when debugging distributed code.")

## Cell 4: The TrainingProfiler

You will paste this class into every module. It measures the four numbers that matter:
throughput, step time, GPU memory, and GPU utilization.

In [ ]:
import time
import torch

class TrainingProfiler:
    """
    Measures and prints key training metrics after each step.
    
    Usage:
        profiler = TrainingProfiler(batch_size=32, rank=0)
        for step, (data, labels) in enumerate(dataloader):
            ... training step ...
            profiler.step(step, loss.item())
        profiler.summary()
    """

    def __init__(self, batch_size: int, rank: int = 0, log_every: int = 10):
        self.batch_size = batch_size
        self.rank = rank
        self.log_every = log_every
        self._step_start = time.perf_counter()
        self._session_start = time.perf_counter()
        self._losses = []
        self._throughputs = []

    def step(self, step: int, loss: float):
        """Call at the end of each training step."""
        if self.rank != 0:
            return  # Only rank 0 prints

        elapsed = time.perf_counter() - self._step_start
        throughput = self.batch_size / elapsed  # samples/second
        self._losses.append(loss)
        self._throughputs.append(throughput)

        if step % self.log_every == 0:
            gpu_info = self._get_gpu_info()
            print(
                f"[Step {step:4d}] "
                f"loss={loss:.4f} | "
                f"step_time={elapsed*1000:.1f}ms | "
                f"throughput={throughput:.0f} samples/s | "
                f"{gpu_info}"
            )

        self._step_start = time.perf_counter()

    def _get_gpu_info(self) -> str:
        parts = []
        for i in range(torch.cuda.device_count()):
            mem_used = torch.cuda.memory_allocated(i) / 1e9
            mem_total = torch.cuda.get_device_properties(i).total_memory / 1e9
            try:
                util = torch.cuda.utilization(i)
                parts.append(f"GPU{i}={mem_used:.1f}/{mem_total:.0f}GB ({util}%)")
            except Exception:
                parts.append(f"GPU{i}={mem_used:.1f}/{mem_total:.0f}GB")
        return " | ".join(parts)

    def summary(self):
        """Print a final summary after training is done."""
        if self.rank != 0 or not self._losses:
            return
        total_time = time.perf_counter() - self._session_start
        avg_throughput = sum(self._throughputs) / len(self._throughputs)
        final_loss = self._losses[-1]
        print()
        print("=" * 60)
        print("TRAINING SUMMARY")
        print(f"  Total time:        {total_time:.1f}s")
        print(f"  Avg throughput:    {avg_throughput:.0f} samples/s")
        print(f"  Final loss:        {final_loss:.4f}")
        print(f"  Steps completed:   {len(self._losses)}")
        print("=" * 60)


# Test the profiler with a fake training loop
print("Testing the profiler with a fake training loop...")
profiler = TrainingProfiler(batch_size=32, rank=0, log_every=2)

for step in range(10):
    time.sleep(0.05)  # simulate work
    fake_loss = 2.0 * (0.9 ** step)
    profiler.step(step, fake_loss)

profiler.summary()

## Cell 5: Understand GPU Memory Layout

This is critical background for Modules 02-04.

In [ ]:
import torch
import torch.nn as nn

def count_parameters(model: nn.Module) -> dict:
    """Count how much memory a model needs for params alone."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # fp32: 4 bytes per param
    # fp16: 2 bytes per param
    param_mem_fp32 = total_params * 4 / 1e9
    param_mem_fp16 = total_params * 2 / 1e9
    
    # In standard DDP training, each GPU holds:
    #   params (fp16):     2 bytes/param
    #   gradients (fp16):  2 bytes/param  
    #   optimizer states:  8 bytes/param  (Adam: m + v, both fp32)
    #   master weights:    4 bytes/param  (fp32 copy for stable updates)
    # Total: ~16 bytes/param
    full_training_mem = total_params * 16 / 1e9
    
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "params_fp32_GB": param_mem_fp32,
        "params_fp16_GB": param_mem_fp16,
        "full_training_mem_GB": full_training_mem,
    }

# Build models of different sizes and see how memory requirements scale
configs = [
    ("Tiny",   dict(d_model=128, nhead=4,  num_layers=2)),
    ("Small",  dict(d_model=256, nhead=8,  num_layers=6)),
    ("Medium", dict(d_model=512, nhead=8,  num_layers=12)),
    ("Large",  dict(d_model=1024, nhead=16, num_layers=24)),
]

print(f"{'Name':<10} {'Params':>12} {'FP16 Params':>12} {'Full Train':>12} {'Fits 1x T4?':>12}")
print("-" * 65)

for name, cfg in configs:
    encoder_layer = nn.TransformerEncoderLayer(
        cfg['d_model'], cfg['nhead'], 
        dim_feedforward=cfg['d_model']*4,
        batch_first=True
    )
    model = nn.Sequential(
        nn.Embedding(50000, cfg['d_model']),
        nn.TransformerEncoder(encoder_layer, cfg['num_layers']),
        nn.Linear(cfg['d_model'], 50000)
    )
    stats = count_parameters(model)
    fits = "Yes" if stats['full_training_mem_GB'] < 16 else "No (needs sharding)"
    print(
        f"{name:<10} "
        f"{stats['total_params']:>12,} "
        f"{stats['params_fp16_GB']:>11.2f}GB "
        f"{stats['full_training_mem_GB']:>11.2f}GB "
        f"{fits:>12}"
    )

print()
print("Note: T4 has 16GB VRAM. 'Full Train' includes params + grads + optimizer state.")
print("Activations (intermediate tensors) add more on top of this.")

## Cell 6: Baseline — Single GPU Training

Run this to get your baseline throughput. Every subsequent module should be faster or use less memory.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time

BATCH_SIZE = 64
SEQ_LEN = 128
VOCAB_SIZE = 10000
D_MODEL = 256
NUM_STEPS = 50
DEVICE = torch.device("cuda:0")

# Model
class BenchmarkModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(
            D_MODEL, nhead=8, dim_feedforward=D_MODEL*4, 
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.classifier = nn.Linear(D_MODEL, 2)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.classifier(x[:, 0, :])  # CLS token classification

# Dataset
data = torch.randint(0, VOCAB_SIZE, (500, SEQ_LEN))
labels = torch.randint(0, 2, (500,))
dataset = TensorDataset(data, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

model = BenchmarkModel().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Warm up (first few steps are slower due to CUDA kernel compilation)
print("Warming up...")
for batch_data, batch_labels in dataloader:
    optimizer.zero_grad()
    out = model(batch_data.to(DEVICE))
    loss = criterion(out, batch_labels.to(DEVICE))
    loss.backward()
    optimizer.step()
    break

# Benchmark
print("Running benchmark...")
profiler = TrainingProfiler(batch_size=BATCH_SIZE, rank=0, log_every=10)
step = 0

for epoch in range(4):
    for batch_data, batch_labels in dataloader:
        if step >= NUM_STEPS:
            break
        optimizer.zero_grad()
        out = model(batch_data.to(DEVICE))
        loss = criterion(out, batch_labels.to(DEVICE))
        loss.backward()
        optimizer.step()
        profiler.step(step, loss.item())
        step += 1

profiler.summary()

print()
print("RECORD THIS NUMBER — it is your baseline.")
print("Every subsequent module should beat it in throughput.")

## Checkpoint

Before moving on, answer these:

1. What does `rank` mean in distributed training?
2. What is `world_size` on your Kaggle instance?
3. What was your baseline throughput (samples/second)?
4. Why does a model require ~16 bytes per parameter during training (not just 2 or 4)?

**Answers:**
1. `rank` is a unique integer ID for each process/GPU. rank=0 is the main process.
2. `world_size` = 2 (you have 2 GPUs)
3. Should be ~200-400 samples/s for the config above. Write it down.
4. fp16 params (2B) + fp16 grads (2B) + fp32 master weights (4B) + Adam m (4B) + Adam v (4B) = 16 bytes/param

---

## Next: Open `../01_data_parallelism/notebook.ipynb`